# **Criptografía post-cuántica y su relevancia en la ciberseguridad**

# *Practiva 1: KEM y DSA usando criptografía convencional*
# 6.° Encuentro de Ingenierías UPAEP 2026

# Objetivos
Comprender cómo se establece un canal seguro en un protocolo criptográfico y cómo esta arquitectura se relaciona con la transición hacia criptografía post-cuántica
Implementar acuerdo de clave con X25519

*   Autenticar el protocolo usando ECDSA
*   Derivar una clave de sesión con HKDF
*   Establecer un canal seguro con AES-GCM
*   Relacionar primitivas clásicas con ML-KEM y ML-DSA

# Resultados esperados
Que el participante:

*   Identifique los componentes de un protocolo seguro
*   Diferencie acuerdo de llave vs firma digital
*   Entienda la transición conceptual hacia criptografía post-cuántica


# Instalación

In [ ]:
!pip install cryptography

In [ ]:
import os

from cryptography.hazmat.primitives.asymmetric import x25519, ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidSignature

# Identidad del servidor (ECDSA)

In [ ]:
print("Identidad del Servidor")
# Servidor genera llaves para identificarse
server_id_private = ec.generate_private_key(ec.SECP256R1())
server_id_public = server_id_private.public_key()

server_id_public_bytes = server_id_public.public_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

print("Curva:", server_id_public.curve.name)
print("Tamaño de la llave pública:", len(server_id_public_bytes), "bytes")

Identidad del Servidor
Curva: secp256r1
Tamaño de la llave pública: 91 bytes


# Llaves usando EC x25519

In [ ]:
print("\nLlaves X25519")
# Cliente genera llaves para el secreto compartido
client_private = x25519.X25519PrivateKey.generate()
client_public = client_private.public_key()

server_private = x25519.X25519PrivateKey.generate()
server_public = server_private.public_key()

#Number used once
nonce_client = os.urandom(16)
nonce_server = os.urandom(16)

client_pub_bytes = client_public.public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw
)

server_pub_bytes = server_public.public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw
)

print("LLave pública cliente:", len(client_pub_bytes), "bytes")
print("Llave pública servidor:", len(server_pub_bytes), "bytes")


Llaves X25519
LLave pública cliente: 32 bytes
Llave pública servidor: 32 bytes


# Construcción del registro para proteger el intercambio

In [ ]:
print("\n Construcción del registro")

transcript = b"|".join([
    client_pub_bytes,
    server_pub_bytes,
    nonce_client,
    nonce_server
])

print("Tamaño del archivo:", len(transcript), "bytes")
print("Primeros bytes:", transcript[:20].hex())


 Construcción del registro
Tamaño del archivo: 99 bytes
Primeros bytes: e785dd71ef29f0ecfe63cde28963b21b1eaf273b


# Firma para el intercambio

In [ ]:
print("\nFirma del registro")

signature = server_id_private.sign(
    transcript,
    ec.ECDSA(hashes.SHA256())
)

print("Tamaño de la firma:", len(signature), "bytes")


Firma del registro
Tamaño de la firma: 71 bytes


# Verificación de Firma del registro



In [ ]:
print("\nVerificación de Firma")

try:
    server_id_public.verify(
        signature,
        transcript,
        ec.ECDSA(hashes.SHA256())
    )

    print("Firma válida")

except InvalidSignature:

    print("Firma inválida")


Verificación de Firma
Firma válida


# Secreto Compartido

In [ ]:
print("\nSecreto Compartido (X25519)")

shared_client = client_private.exchange(server_public)
shared_server = server_private.exchange(client_public)

print("Longitud del secreto:", len(shared_client), "bytes")
print("¿Coinciden?:", shared_client == shared_server)


Secreto Compartido (X25519)
Longitud del secreto: 32 bytes
¿Coinciden?: True


# Derivación de la llave


In [ ]:
print("\n Derivación de la llave")

hkdf = HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=None,
    info=b"Registro|" + transcript
)

session_key = hkdf.derive(shared_client)

print("Llave de sesión:", len(session_key), "bytes")
print("Primeros bytes:", session_key[:10].hex())


 Derivación de la llave
Llave de sesión: 32 bytes
Primeros bytes: c7e19c5cd215a6a27730


# Intercambio seguro

In [ ]:
print("\nCanal seguro")

aesgcm = AESGCM(session_key)

message = b"Hola, canal seguro autenticado"
nonce = os.urandom(12)

ciphertext = aesgcm.encrypt(nonce, message, None)

plaintext = aesgcm.decrypt(nonce, ciphertext, None)

print("Mensaje original:", message)
print("Mensaje descifrado:", plaintext)


Canal seguro
Mensaje original: b'Hola, canal seguro autenticado'
Mensaje descifrado: b'Hola, canal seguro autenticado'


# Demostración de integridad

In [ ]:
print("\nPrueba de Integridad")

tampered = transcript[:-1] + bytes([transcript[-1] ^ 0x01])

try:

    server_id_public.verify(
        signature,
        tampered,
        ec.ECDSA(hashes.SHA256())
    )

    print("Firma valida")

except InvalidSignature:

    print("Firma invalida")


Prueba de Integridad
Firma invalida


Ahora que ya vimos una introducción a la criptografía clásica y a la criptografía post-cuántica, vamos a pasar a una pequeña demostración práctica.

Para ello utilizaremos un laboratorio interactivo en Google Colab, donde podrán ejecutar directamente algunos algoritmos post-cuánticos como ML-KEM y ML-DSA sin instalar nada en sus equipos.

Por favor abran el siguiente enlace y ejecuten las primeras celdas hasta el paso 3, ya que el proceso de instalación y compilación puede tardar algunos minutos.

https://colab.research.google.com/drive/1mXfUKMKBrPCQiWE5yt7wJj-zXwNNvM31